In [ ]:
import os
if not hasattr(os, "_cwd"):
    os.chdir("../../")
    os._cwd = True

In [ ]:
import os
import sys
import base64
import hashlib

from rich import print

from Environment import *


class EncryptDecrypt:
    def __init__(self, password=SECRET_KEY):
        self.password = password
        self.key_hash = hashlib.sha256(password.encode()).hexdigest()

    def encrypt(self, text, length=128):
        # XOR encryption
        result = ""
        for i, char in enumerate(text):
            result += chr(ord(char) ^ ord(self.key_hash[i % len(self.key_hash)]))

        # Expand/truncate to fixed length using hash
        if len(result) < length:
            expansion = hashlib.sha256((result + self.password).encode()).hexdigest()
            while len(result) < length:
                result += expansion
                expansion = hashlib.sha256(expansion.encode()).hexdigest()

        # Truncate to exact length
        result = result[:length]

        # Prepend the length of the original text as a fixed-size header (e.g., 4 bytes)
        header = f"{len(text):04d}"
        result_with_header = header + result
        return base64.b64encode(result_with_header.encode()).decode()

    def decrypt(self, encrypted_text):
        decoded = base64.b64decode(encrypted_text.encode()).decode()

        # Extract the original length from the header
        original_length = int(decoded[:4])
        encrypted_portion = decoded[4 : 4 + original_length]

        result = ""
        for i, char in enumerate(encrypted_portion):
            result += chr(ord(char) ^ ord(self.key_hash[i % len(self.key_hash)]))

        return result


se = EncryptDecrypt()

data = "Hello, World!"

print("Original Data:", data)

print("Encrypted Data:", se.encrypt(data))

print("Decrypted Data:", se.decrypt(se.encrypt(data)))



Original Data: Hello, World!

Encrypted Data: 
MDAxM3EBDg9WGBYyWRZaVBc5NjEwMGNmZjYzYWEwNWVkYTUwNzA3ODNhZjQwYWRlN2NhNmRiNjI4YTQ2OGJjMDU0YjY2MTZjOGNiNGFkYWU0Mjc4MDY
4Y2E0YzY3ZjYxNGU3ODA3YjUxMGIyNGI4ODYxNDkzMWEzOTUzN2Y1MzMzNzJk

Decrypted Data: Hello, World!